<a id="unit-u.08bcc5b5f71447b4955bee5ab3956e19"></a>

<div>
  <div>2-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН ДЕРЕКТЕР ҚОЙМАСЫН ҚҰРУ</div>
  <div>Сақтауды және жазу пакеттерін бақылау</div>
</div>

1-модульдегі он тапсырысты анықтамасы бірдей екі кестеге жазыңыз: біреуіне он жолдан тұратын бір пакет ретінде, екіншісіне бір жолдан он жазу ретінде. Екі кесте бірдей бизнес нәтижесін беруі керек; содан кейін екі әдістің сақтауда не қалдырғанын, tablet нұсқалары мен rowset-ін қарайсыз және сұраудың не жоспарлағанын, ал сканерлеудің іс жүзінде не оқығанын көру үшін `EXPLAIN` мен Query Profile қолданасыз. Шамамен 20 минут жоспарлаңыз.

**2-модуль: Doris сақтау архитектурасы және жазу механизмдері** · [Сабақ](course.md) · [Тест](quiz02_storage_and_write_batches.ipynb) · [Курстың README файлы](../../README.md)


<a id="unit-u.f253021b42a64e82a8a6f6810b68401a"></a>

<a id="lab-scope"></a>

## Зертханалық жұмыстың қамту аясы

Бұл жұмыс зертханалық дерекқорыңыздағы тек `orders_batch` және `orders_rowwise` кестелерін қайта жасайды. 1-зертханалық жұмыстың курс сынақ ортасы қажет, басқа кесте керек емес. Фондық compaction rowset-терді қайта ұйымдастыруды жалғастырады, сондықтан сақтаудың кей мәліметтері қараған уақытқа байланысты; әр «Күтілетін нәтиже» қайсысы екенін көрсетеді.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect
from dw_course.wwi import HISTORY_COLUMNS as ORDER_COLUMNS, history_ddl as order_ddl, history_rows
from dw_course.ui import show_sql, show_records

lab = connect_sandbox()

<a id="unit-u.f387f936c4274b59b10ef2038e2d4572"></a>

<a id="1-write-the-same-ten-orders-in-two-ways"></a>

## 1. Бірдей он тапсырысты екі тәсілмен жазу

Екі кестеде бірдей бағандар, бір бакет және бір реплика бар; тек жазу тәсілі өзгеше. `orders_batch` он жолдық бір `INSERT`, ал `orders_rowwise` әрқайсысы бір жолдық он `INSERT` операторын алады. Әр `INSERT` — бір жүктеу транзакциясы: ол бекітілгенде, жолдары көрінеді және олар орналасқан tablet-ке бір **нұсқа** қосады.

Қосылым осы сеанс үшін `group_commit = 'off_mode'` орнатқан, сондықтан әр оператор жеке бекітіледі. 5-модуль осындай көптеген шағын жазуды бір транзакцияға біріктіре алатын Group Commit механизмін көрсетеді.


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_batch")
lab.execute("DROP TABLE IF EXISTS orders_rowwise")
ddl = order_ddl("orders_batch")
show_sql("Both tables: orders_rowwise differs only in its name", ddl)
lab.execute(ddl)
lab.execute(order_ddl("orders_rowwise"))

columns = ", ".join(ORDER_COLUMNS)
values = [f"({order_id}, {customer_id}, '{order_date}', {amount}, {lines}, '{source}')"
          for order_id, customer_id, order_date, amount, lines, source in history_rows()]
batch = f"INSERT INTO orders_batch ({columns}) VALUES\n" + ",\n".join(values)
show_sql("One write of ten rows", batch)
lab.execute(batch)
single_rows = [f"INSERT INTO orders_rowwise ({columns}) VALUES {value}" for value in values]
show_sql("Ten writes of one row: the first two", ";\n".join(single_rows[:2]) + ";\n-- ... and eight more, one per order")
for statement in single_rows:
    lab.execute(statement)

<a id="unit-u.425866387a0f48e38f173540826b6fff"></a>

**Күтілетін нәтиже**

Кесте анықтамасы, он жолы бар пакеттік `INSERT` және он біржолдық оператордың алғашқы екеуі. Басқа ештеңе шықпайды: оператор сәтті орындалса, `lab.execute` ештеңе хабарламайды.


<a id="unit-u.1ad27c64a80d45158d0ba5cfadccebf0"></a>

<a id="2-check-the-business-result-first"></a>

## 2. Алдымен бизнес нәтижесін тексеру

Сақтауды салыстырмас бұрын екі жазу тәсілі де бірдей тапсырыстарды сақтағанын тексеріңіз: 1-модульдегідей бірдей жол саны және салыққа дейінгі сома — 10 жол және 12220.60. Сақтаудың өзгеше болуы ешқашан жауаптың өзгеше болуын білдірмеуі керек.


In [ ]:
for table in ("orders_batch", "orders_rowwise"):
    expect(lab.query(f"SELECT COUNT(*), SUM(order_amount) FROM {table}"), [(10, "12220.60")],
           title=f"{table}: ten orders, 12220.60")
lab.sql("""SELECT 'batch' AS write_mode, COUNT(*) AS orders, SUM(order_amount) AS amount FROM orders_batch
UNION ALL
SELECT 'rowwise', COUNT(*), SUM(order_amount) FROM orders_rowwise
ORDER BY write_mode""", title="Business results of both write methods");

<a id="unit-u.c05350014bf1481489321fb466439aa7"></a>

**Күтілетін нәтиже**

Екі жасыл белгі және екі жол — `batch` және `rowwise`, әрқайсысында 10 тапсырыс және 12220.60.


<a id="unit-u.56179b9795414225845223aa69fc851f"></a>

<a id="3-versions-in-the-partition-and-the-tablet"></a>

## 3. Бөлім мен tablet нұсқалары

`SHOW PARTITIONS` және `SHOW TABLETS` FE метадеректерін оқиды. `PARTITION BY` жоқ кестенің де кестемен аттас бір бөлімі бар, ал `BUCKETS 1` сол бөлімге бір tablet береді. Бөлімнің `VisibleVersion` және tablet-тің `Version` өрістері бекітілген жүктеулерді санайды: бос tablet 1-нұсқадан басталып, әр жүктеу бір нұсқа қосады.


In [ ]:
for table in ("orders_batch", "orders_rowwise"):
    lab.sql(f"SHOW PARTITIONS FROM {table}", title=f"The partition of {table}",
            columns=["PartitionName", "VisibleVersion", "Buckets", "ReplicationNum"])
    lab.sql(f"SHOW TABLETS FROM {table}", title=f"The tablet of {table}",
            columns=["TabletId", "Version", "VisibleVersionCount", "VersionCount", "RowCount"])

<a id="unit-u.57878c6714ee4b60a024215e42799048"></a>

**Күтілетін нәтиже**

Әр кестеде бір бакеті және бір репликасы бар бір бөлім және бір tablet бар. `orders_batch` нұсқасы 2: 1-нұсқаға бір жүктеу қосылады. `orders_rowwise` нұсқасы 11: 1-нұсқаға он жүктеу қосылады. Бір жолдан он жазу он транзакция жасаса, бір пакет бір транзакция жасады.

`VisibleVersionCount`, `VersionCount` және `RowCount` шамамен минутына бір рет келетін BE tablet есебінен алынады: жазулардан кейін бірден олар `-1`, `-1` және `0`, ал бір минуттан кейін нақты сандарды көрсетеді. `Version` — FE-нің өз есебі және бірден дұрыс болады.


<a id="unit-u.02c051107b3e46bdaa5095d56ff7373e"></a>

<a id="4-rowsets-and-what-compaction-does-with-them"></a>

## 4. Rowset және compaction оларды қалай өзгертеді?

Әр жүктеу tablet ішіне **rowset** жазады: бір не бірнеше **сегмент** файлындағы нұсқаланған жолдар жиынын. `SHOW TABLET <id>` tablet қайда жататынын анықтап, репликаларын `CompactionStatus` URL-імен тізетін `SHOW PROC` операторы — `DetailCmd` береді. BE-дегі сол URL tablet-тің rowset-терін қайтарады. `lab.rowsets(table)` кестенің әр tablet-і үшін оны оқып, сұрауды көрсетеді; URL контейнер ішіндегі BE мекенжайын, `127.0.0.1:8040`, атайды, сондықтан сол жол хосттың BE порты — 51040 арқылы оқылады.

**Compaction** шағын rowset-терді фондық режимде үлкендеріне біріктіреді; содан кейін сұраулар азырақ файл оқиды. Ол жолдардың сақталу тәсілін өзгертеді, бірақ сұрау қайтаратын жолдарды ешқашан өзгертпейді.


In [ ]:
tablet = lab.query("SHOW TABLETS FROM orders_rowwise")[0][0]
detail = lab.sql(f"SHOW TABLET {tablet}", title="Where the tablet of orders_rowwise belongs",
                 columns=["DbName", "TableName", "PartitionName", "IndexName", "DetailCmd"])
lab.sql(detail["DetailCmd"][0].rstrip(";"), title="Its replica, from the DetailCmd",
        columns=["ReplicaId", "BackendId", "Version", "State", "CompactionStatus"])
lab.rowsets("orders_batch")
lab.rowsets("orders_rowwise");

<a id="unit-u.56499d2791774c61b3361c839b5959fb"></a>

**Күтілетін нәтиже**

Tablet `orders_rowwise` кестесіне жатады, ал оның жалғыз BE-дегі репликасының нұсқасы 11, күйі `NORMAL`, `CompactionStatus` URL-і бар.

`orders_batch` кестесінде екі ағымдағы rowset бар: 0 сегменті бар `[0-1]` — әр tablet-тің бос бастапқы күйі және 1 сегменті, шамамен 1.5 KB көлемі бар `[2-2]` — пакеттің он жолы.

`orders_rowwise` үш сәттің бірін көрсетеді. Жазулардан кейін бірден әрқайсысы шамамен 1.5 KB болатын `[2-2]` мен `[11-11]` арасындағы он ағымдағы rowset көрінеді. Әдетте бір-екі секундтан кейін жинақтаушы compaction оларды біріктіреді: бір ағымдағы `[2-11]` rowset-і және ескірген ретінде тізілген он ескі rowset, оларды BE бірнеше минуттан кейін жояды. Одан кейін тек `[0-1]` және `[2-11]` қалады. `[2-11]` — он тапсырыс емес, нұсқалар ауқымы. Үш сәттің бәрінде 2-қадамдағы сұрау бірдей он тапсырысты қайтарады.


<a id="unit-u.e90fee07017842568e0e106a3b6e17a7"></a>

<a id="5-the-plan-and-the-work-explain-and-the-query-profile"></a>

## 5. Жоспар және орындалған жұмыс: EXPLAIN және Query Profile

`EXPLAIN` FE сұрауға құрған жоспарды орындамай көрсетеді: сканерленетін кестені, бағандарды және сүзгіні. **Query Profile** сұрау орындалғанда әр оператор іс жүзінде не істегенін тіркейді: сканерлеу қанша жол оқыды, қаншасын және қалай өткізіп жіберді, қанша байт оқыды.

Келесі ұяшық осы сеанс үшін Profile-дарды қосып, қайталанған сұрау шынымен сканерлеуі үшін SQL кэшін өшіреді және кейін екі параметрді де қалпына келтіреді. `lab.scan_profile(query)` сұрауды орындайды, `SELECT last_query_id()` арқылы сұрау идентификаторын алады (сұраудың Profile ID-і — оның сұрау ID-і), FE-ден Profile мәтінін жүктеп, сканерлеу жолдарының есептегіштерін қосады. Ол сұрыптау кілті `order_id` болатын `orders_batch` кестесінде үш сұрауды салыстырады:

| Есептегіш | Мағынасы |
| --- | --- |
| `ScanRows` | Сканерлеу сегменттерінен оқыған жолдар |
| `RowsKeyRangeFiltered` | Сұрыптау кілтінің индексі сканерлеуге оқымай өткізіп жіберуге мүмкіндік берген жолдар |
| `RowsStatsFiltered` | Zone map арқылы, яғни әр бет пен сегменттің ең кіші және ең үлкен мәндері арқылы өткізіп жіберілген жолдар |
| `RowsVectorPredFiltered` | Оқылып, кейін сүзгі алып тастаған жолдар |
| `RowsRead` | Сүзгіден өткен жолдар: `ScanRows` минус `RowsVectorPredFiltered` |

FE Web UI сол Profile-дарды көрсетеді: `http://127.0.0.1:51030` мекенжайын (немесе `DW_FE_HTTP_PORT` ішіндегі портты) ашып, бос құпиясөзбен `root` ретінде кіріңіз және **QueryProfile** таңдаңыз.


In [ ]:
lab.sql("EXPLAIN SELECT order_id, order_amount FROM orders_batch WHERE order_id = 1",
        title="The plan: the table, the columns, and the filter");
previous = lab.query("SELECT @@enable_profile, @@enable_sql_cache")[0]
lab.execute("SET enable_profile = true, enable_sql_cache = false")
try:
    scans = [lab.scan_profile(statement) for statement in (
        "SELECT order_id, order_amount FROM orders_batch WHERE order_id = 1",
        "SELECT order_id, order_amount FROM orders_batch WHERE customer_id = 832",
        "SELECT * FROM orders_batch WHERE customer_id = 832",
    )]
finally:
    lab.execute("SET enable_profile = %s, enable_sql_cache = %s", previous)
show_records("What the scan of each query read", scans)

<a id="unit-u.401f01921baf4f9fbd35ed27d5afe829"></a>

**Күтілетін нәтиже**

Жоспардың сканерлеу түйінінде `orders_batch`, `order_id = 1` предикаты және `partitions=1/1`, `tablets=1/1` бар: бұл оқылғанның саны емес, жоспар. Содан кейін сұрау идентификаторлары бар үш оператор, олардың Profile-дарына арналған curl сұраулары және әр сұрауға бір жолы бар кесте көрсетіледі:

| Сұрау | rows_returned | ScanRows | RowsKeyRangeFiltered | RowsVectorPredFiltered |
| --- | ---: | ---: | ---: | ---: |
| `WHERE order_id = 1`, екі баған | 1 | 1 | 9 | 0 |
| `WHERE customer_id = 832`, екі баған | 1 | 10 | 0 | 9 |
| `WHERE customer_id = 832`, барлық баған | 1 | 10 | 0 | 9 |

Қорытындыда әр `query_id` және оның `profile_url` мәні бар; дәл сол FE Profile-ды ашу үшін астындағы сәйкес ID-ді басыңыз (`root` пайдаланушысы, бос құпиясөз). Сілтеме сіз баптаған FE HTTP портын қолданады. Кіру FE басты бетін ашса, Profile-ға оралу үшін сол ID-ді тағы басыңыз. Қайта орындағаннан кейін Profile тізімінен ұқсас SQL арқылы таңдаудың орнына жаңа жолдың ID-ін пайдаланыңыз. Есептегіштер өзгеше болса, ID мен Profile бастапқы мәтінін сақтап, BE есеп беруді аяқтағаннан кейін сол Profile-ды қайта оқыңыз.

Үшеуі де 1-тапсырысты қайтарады, бірақ әртүрлі жұмыс істейді. Сұрыптау кілтінің бағанына қойылған сүзгі индекске сканерлеуді бір жолға тарылтуға мүмкіндік береді. Кілтке кірмейтін `customer_id` сүзгісі он жолдың бәрін оқытып, тоғызын алып тастайды. Бірдей сүзгімен үшінші сұрау екі бағанның орнына алты бағанның бәрін оқиды, сондықтан оның `ScanBytes` мәні айқын жоғары, бірнеше жүз байтқа артық: бағандық сақтау тек сұрау қолданатын бағандарды оқиды. Он жолда бұл көлемдер өте аз және уақыттар жылдамдық туралы ештеңе айтпайды; есептегіштер сұраудың қандай жұмыс істейтінін көрсетеді, сол есептегіштер деректермен бірге өседі.


<a id="unit-u.0deba8bc56cb4ed0b8f100317e7b32c0"></a>

<a id="6-practice-one-date-in-both-tables"></a>

## 6. Жаттығу: екі кестедегі бір күн

Екі кестедегі 2013-01-01 тапсырыстарын сұрап, бір нәтижеде көрсетіңіз; әр кесте үшін 5 тапсырыс және 3944.20 күтіледі. Содан кейін 5-қадамдағыдай Profile-дарды қосып, `SELECT COUNT(*), SUM(order_amount) FROM orders_batch WHERE order_date = '2013-01-01'` сканерлеу есептегіштерін оқыңыз және сандарды түсіндіріңіз: сұрыптау кілті бұл сүзгіге көмектесе ме?


In [ ]:
# Your code here

<a id="unit-u.d3c95e8681d84585bfbcb30a08740b30"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған күйде. Салыстыру үшін ашыңыз немесе орындаңыз.


In [ ]:
query = """SELECT 'batch' AS write_mode, COUNT(*) AS orders, SUM(order_amount) AS amount
FROM orders_batch WHERE order_date = '2013-01-01'
UNION ALL
SELECT 'rowwise', COUNT(*), SUM(order_amount)
FROM orders_rowwise WHERE order_date = '2013-01-01'
ORDER BY write_mode"""
lab.sql(query, title="First-day orders from both write methods");
expect(lab.query(query), [("batch", 5, "3944.20"), ("rowwise", 5, "3944.20")],
       title="Both tables: five orders, 3944.20")
previous = lab.query("SELECT @@enable_profile, @@enable_sql_cache")[0]
lab.execute("SET enable_profile = true, enable_sql_cache = false")
try:
    show_records("What the scan of the date filter read", [lab.scan_profile(
        "SELECT COUNT(*), SUM(order_amount) FROM orders_batch WHERE order_date = '2013-01-01'")])
finally:
    lab.execute("SET enable_profile = %s, enable_sql_cache = %s", previous)

<a id="unit-u.a8c6952eb69549a9b92a630fe99f6d73"></a>

**Күтілетін нәтиже**

`batch` және `rowwise`, әрқайсысында 5 тапсырыс және 3944.20. Күн сүзгісінің сканерлеуі: `ScanRows` 10, `RowsRead` 5, `RowsVectorPredFiltered` 5; кілт ауқымы да, zone map та ештеңені өткізіп жібермейді. `order_date` сұрыптау кілті `order_id` құрамына кірмейді, сондықтан индекс сканерлеуді тарылта алмайды, ал екі күн бір сегментте болғандықтан, оның ең кіші және ең үлкен күндері де сегментті алып тастауға мүмкіндік бермейді. 3-модуль тапсырыстарды күн бойынша бөлімдейді, сондықтан бір күннің сұрауы екінші күннің деректерін өткізіп жібереді.


<a id="unit-u.b3a7ea0b2dce492da7eb1095d3bf7e58"></a>

<a id="you-are-done-when"></a>

## Жұмысты аяқтау шарттары

- екі кестеде де 10 тапсырыс және 12220.60 бар, әрі нұсқалары неге өзгеше екенін айта аласыз: бір пакет үшін 2, он жазу үшін 11;
- екі кестенің rowset-ін қарап, `[2-11]` нені білдіретінін және compaction нені өзгерткенін түсіндіре аласыз;
- жоспардан сұрау не оқуды көздейтінін, ал Profile-дан сканерлеу не оқып, нені өткізіп жібергенін анықтай аласыз.

Он жол — оқу үлгісі: ол өндірістік жылдамдықты емес, механизмдерді көрсетеді. Жазудың өткізу қабілетін өлшеу үшін үлкен деректер, тұрақты параллельдік және уақыт бойы бақылау қажет.


<a id="unit-u.4f11551d73e5443e9d47af7e71ed4c3a"></a>

<a id="next"></a>

## Келесі қадам

[2-тест: Doris сақтау архитектурасы және жазу механизмдері](quiz02_storage_and_write_batches.ipynb) файлын ашыңыз, содан кейін [3-модуль: Doris кесте модельдері, бөлімдеу және бакеттерге бөлу](../module03-table-design/course.md) бөліміне өтіңіз.
